# Stage 3 — Mine the predicate edges (and draw a fresh validation sample)

Read each summary PDF's text and apply the **deterministic edge rule**: in any summary that
contains a predicate cue (*predicate*, *substantially equivalent*, *reference device*), every
**other in-corpus K-number** mentioned becomes an incoming predicate edge for that device.
Each edge is flagged `SECTION_HEADED` (a cue within 300 characters of the K-number) or
`PROXIMITY_ONLY`.

It also draws a **fresh random sample of 60 devices** (D3) for hand-adjudication of precision.

Network: none — reads the PDFs downloaded in Stage 2. Requires `pymupdf` (`import fitz`).

In [ ]:
import os, re, gzip, json, random
import pandas as pd
import fitz  # pymupdf

corpus = pd.read_csv("data/corpus.csv", dtype=str)
VALID = set(corpus["k_number"])
KPAT = re.compile(r"\bK\d{6}\b")
CUE = re.compile(r"(?i)predicate|substantial(?:ly)?\s+equival|reference\s+device")
CUE_STRONG = re.compile(r"(?i)predicate|reference\s+device")

man = pd.read_csv("data/download_manifest.csv", dtype=str)
have = man[man["status"] == "200"]["k_number"].tolist()
print(f"{len(have)} PDFs with text to mine")

### Extract text once, cache to disk

In [ ]:
os.makedirs("data/text", exist_ok=True)
def get_text(k):
    cache = f"data/text/{k}.txt"
    if os.path.exists(cache):
        return open(cache, encoding="utf-8").read()
    try:
        doc = fitz.open(f"data/summaries/{k}.pdf")
        txt = "\n".join(p.get_text() for p in doc)
        doc.close()
    except Exception:
        txt = ""
    open(cache, "w", encoding="utf-8").write(txt)
    return txt

### Apply the deterministic rule

In [ ]:
edges = []
coverage = []
for k in have:
    txt = get_text(k)
    has_cue = bool(CUE.search(txt))
    found = [m for m in KPAT.findall(txt)]
    in_corpus = sorted({x for x in found if x in VALID and x != k})
    # coverage diagnostic
    if not has_cue:
        cov = "no_cue"
    elif in_corpus:
        cov = "edge_resolvable"
    elif found:
        cov = "out_of_scope"      # cites K-numbers, but none in our corpus
    else:
        cov = "name_only"         # cue present, but no K-number at all
    coverage.append({"k_number": k, "coverage": cov})
    if has_cue:
        for pred in in_corpus:
            # confidence: cue within 300 chars of this K-number's first mention?
            idx = txt.find(pred)
            window = txt[max(0, idx-300): idx+300]
            conf = "SECTION_HEADED" if CUE_STRONG.search(window) else "PROXIMITY_ONLY"
            edges.append({"predicate_knumber": pred, "device_knumber": k, "confidence": conf})

edf = pd.DataFrame(edges).drop_duplicates()
edf.to_csv("data/predicate_edges.csv", index=False)
cov = pd.DataFrame(coverage)
cov.to_csv("data/coverage_diagnostic.csv", index=False)

n = len(have)
cc = cov["coverage"].value_counts()
print(f"CHECKPOINT  edges {len(edf)} | devices with >=1 edge {edf['device_knumber'].nunique()}")
print(f"CHECKPOINT  confidence {edf['confidence'].value_counts().to_dict()}")
print(f"CHECKPOINT  coverage: cue {100*(n-cc.get('no_cue',0))/n:.1f}% | "
      f"resolvable {100*cc.get('edge_resolvable',0)/n:.1f}% | "
      f"name-only {100*cc.get('name_only',0)/n:.1f}% | "
      f"out-of-scope {100*cc.get('out_of_scope',0)/n:.1f}%")

### Draw the fresh 60-device validation sample (D3)

This writes `data/validation_sample.csv` — 60 randomly chosen devices with their pipeline-found
predicates. Adjudicate by reading each device's PDF and confirming every listed K-number is a
genuine predicate/reference citation (precision), then record the result in the validation report.

In [ ]:
random.seed(42)
sample = random.sample(have, 60)
vs = edf[edf["device_knumber"].isin(sample)].copy()
vs.to_csv("data/validation_sample.csv", index=False)
print(f"CHECKPOINT  validation sample: 60 devices, {len(vs)} candidate edges to hand-adjudicate")
print("  -> open each device's data/text/<k>.txt, confirm every predicate_knumber is a real citation")